# Notebook 01 — Data Loading

**Goal:** Load the two CSV datasets, inspect structure, document column types and null counts.

**Datasets:**
- `data/raw/marathon/all_marathons_final.csv` — Marathon Results 2000–2019 (Zenodo 6959864)
- `data/raw/athletes/run_ww_2019_w.csv` — Long-Distance Running, weekly aggregation (Kaggle: mexwell)

The Kaggle dataset also ships `run_ww_2019_m.csv` / `run_ww_2019_q.csv` (the **same data** aggregated monthly/quarterly) and three `covid-*` policy-index files with a different schema. Only the weekly running file is used; the others are ignored.


In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = '../data/raw'
MARATHON_CSV = os.path.join(RAW_DIR, 'marathon', 'all_marathons_final.csv')
ATHLETES_CSV = os.path.join(RAW_DIR, 'athletes', 'run_ww_2019_w.csv')

print('Marathon file exists:', os.path.exists(MARATHON_CSV))
print('Athlete file exists: ', os.path.exists(ATHLETES_CSV))

Marathon file exists: True
Athlete file exists:  True


## 0. Download Raw Data (run once)

Files are fetched only if missing, so this cell is a no-op once the data is in place.

- **Zenodo** (marathon results) is a direct public download.
- **Kaggle** (Strava running data) requires an API access token in the `KAGGLE_API_TOKEN` environment variable (kaggle.com → Settings → API → Create New Token). Only `run_ww_2019_w.csv` is downloaded — the dataset's other 12 files (2020 data, daily/monthly/quarterly granularities, covid indexes) are not used.

In [7]:
# Fetch the raw files if they are missing. The token is read from the
# environment — never paste it into the notebook.
import subprocess
import urllib.request

ZENODO_URL = 'https://zenodo.org/api/records/6959864/files/all_marathons_final.csv/content'
KAGGLE_DATASET = 'mexwell/long-distance-running-dataset'

if os.path.exists(MARATHON_CSV):
    print('Marathon CSV already present — skipping download.')
else:
    os.makedirs(os.path.dirname(MARATHON_CSV), exist_ok=True)
    print('Downloading all_marathons_final.csv (354 MB) from Zenodo...')
    urllib.request.urlretrieve(ZENODO_URL, MARATHON_CSV)
    print('Done.')

if os.path.exists(ATHLETES_CSV):
    print('Athlete CSV already present — skipping download.')
else:
    if not os.environ.get('KAGGLE_API_TOKEN'):
        raise RuntimeError(
            'Set the KAGGLE_API_TOKEN environment variable first '
            '(kaggle.com → Settings → API → Create New Token).'
        )
    os.makedirs(os.path.dirname(ATHLETES_CSV), exist_ok=True)
    print('Downloading run_ww_2019_w.csv (149 MB) from Kaggle...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', KAGGLE_DATASET,
         '-f', os.path.basename(ATHLETES_CSV), '--unzip',
         '-p', os.path.dirname(ATHLETES_CSV)],
        check=True,
    )
    print('Done.')

Marathon CSV already present — skipping download.
Athlete CSV already present — skipping download.


## 1. Marathon Results Dataset

In [ ]:
# Load the single marathon CSV from the Zenodo record.
if not os.path.exists(MARATHON_CSV):
    raise FileNotFoundError('all_marathons_final.csv not found — run the download cell above.')

marathon_raw = pd.read_csv(MARATHON_CSV, low_memory=False)
print(f'Total marathon rows: {len(marathon_raw):,} — {marathon_raw.shape[1]} cols')

In [2]:
print('=== Marathon columns ===')
print(marathon_raw.dtypes)
print()
marathon_raw.head(3)

=== Marathon columns ===


NameError: name 'marathon_raw' is not defined

In [ ]:
print('=== Null counts ===')
nulls = marathon_raw.isnull().sum()
print(nulls[nulls > 0])
print(f'\nNull percentage per column:')
print((nulls[nulls > 0] / len(marathon_raw) * 100).round(2))

In [ ]:
print('=== Summary statistics ===')
marathon_raw.describe(include='all')

In [ ]:
# Inspect the schema: print unique values for categorical columns.
for col in marathon_raw.columns:
    n_unique = marathon_raw[col].nunique()
    if n_unique < 30:
        print(f'{col} ({n_unique} unique): {sorted(marathon_raw[col].dropna().unique().tolist())}')
    else:
        sample = marathon_raw[col].dropna().head(3).tolist()
        print(f'{col} ({n_unique} unique): sample = {sample}')

In [ ]:
# Columns chosen by hand from the header inspected above:
#   finish time → 'Time'  (decimal minutes; 'Gun Time' is the same info as H:MM:SS)
#   age         → 'Age'   ('Agegroup' is the race's own, inconsistent division label)
#   gender      → 'Sex'   ('Sex1' is a messy duplicate, dropped in notebook 02)
KEY_COLS = ['Time', 'Age', 'Sex', 'Year', 'Marathon']

missing = [c for c in KEY_COLS if c not in marathon_raw.columns]
assert not missing, f'Schema changed — missing columns: {missing}'
print('All key columns present:', KEY_COLS)

## 2. Athlete Biometrics Dataset

In [3]:
# Load only the weekly running file. The monthly/quarterly files duplicate this
# data at coarser granularity, and the covid-* files have a different schema —
# concatenating them would mix incompatible tables and triple-count athletes.
if not os.path.exists(ATHLETES_CSV):
    raise FileNotFoundError('run_ww_2019_w.csv not found — run the download cell above.')

athletes_raw = pd.read_csv(ATHLETES_CSV, index_col=0)
print(f'Athlete rows: {len(athletes_raw):,}')
athletes_raw.head(3)

Athlete rows: 1,893,424


,datetime,athlete,distance,duration,gender,age_group,country,major
0,2019-01-01,0,0.00,0.0,F,18 - 34,United States,CHICAGO 2019
1,2019-01-01,1,5.27,30.2,M,35 - 54,Germany,BERLIN 2016
2,2019-01-01,2,9.30,98.0,M,35 - 54,United Kingdom,"LONDON 2018,LONDON 2019"


In [4]:
print('=== Athlete columns ===')
print(athletes_raw.dtypes)
print()
print('=== Null counts ===')
nulls = athletes_raw.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls')

=== Athlete columns ===
datetime         str
athlete        int64
distance     float64
duration     float64
gender           str
age_group        str
country          str
major            str
dtype: object

=== Null counts ===
country    17108
dtype: int64


In [5]:
athletes_raw.describe()

,athlete,distance,duration
count,1.893424e+06,1.893424e+06,1.893424e+06
mean,1.878141e+04,2.924142e+01,1.605819e+02
std,1.081663e+04,3.010521e+01,1.665210e+02
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,9.428750e+03,1.630000e+00,1.075000e+01
50%,1.879250e+04,2.160000e+01,1.229167e+02
75%,2.810325e+04,4.618000e+01,2.545500e+02
max,3.759800e+04,7.111400e+02,8.239200e+03
